In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
import os, gc, shutil
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl

In [2]:
IS_SAMPLE      = False
NEGATIVE_RATIO = 4
CHUNK_SIZE     = 3000000

PROCESSED_DATA_DIR = '/kaggle/input/datasets/b22dckh072/file05'
TRAIN_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(PROCESSED_DATA_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(PROCESSED_DATA_DIR, 'candidates_phase2.parquet')
TEST_PATH = os.path.join(PROCESSED_DATA_DIR, 'test_interactions.parquet')
FEAT_OUT   = os.path.join('/kaggle/working/features.parquet')

In [3]:
print("Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)
lf_cands = pl.scan_parquet(CAND_PATH)

print("Bước 2: Tính toán đặc trưng thống kê & Trọng số...")
# 1. Đặc trưng User
user_stats = lf_train.group_by('mapped_user_id').agg([
    pl.len().cast(pl.Float32).alias('user_total_actions'),
    pl.col('rating').mean().cast(pl.Float32).alias('user_avg_rating_given')
])

# 2. Đặc trưng Item
item_stats = lf_train.group_by('mapped_item_id').agg([
    pl.len().cast(pl.Float32).alias('item_total_sales'),
    pl.col('rating').mean().cast(pl.Float32).alias('item_actual_avg_rating'),
    pl.col('verified_purchase').cast(pl.Float32).sum().alias('item_verified_sales'),
    pl.col('helpful_vote').cast(pl.Float32).sum().alias('item_raw_helpful_votes')
]).with_columns([
    (pl.col('item_verified_sales') / pl.col('item_total_sales')).fill_null(0.0).alias('item_verified_ratio'),
    (pl.col('item_raw_helpful_votes') + 1.0).log().alias('item_log_helpful_votes')
]).drop(['item_verified_sales', 'item_raw_helpful_votes']) 

print("Bước 3: Tối ưu bộ nhớ Meta Data...")
lf_meta = lf_meta.with_columns([
    pl.col('store').cast(pl.Utf8).fill_null('Unknown').cast(pl.Categorical),
    pl.col('categories').cast(pl.Utf8)
      .str.replace_all(r"\[|\]|'|\"", "") 
      .str.split(',')
      .list.get(2) # Lấy danh mục Cấp 3 thay vì last()
      .str.strip_chars()
      .fill_null('Unknown')
      .cast(pl.Categorical)
      .alias('main_category')
])

print("Bước 4: Đánh nhãn trực tiếp trên tập Ứng viên (Chống Data Leakage)...")
lf_test = pl.scan_parquet(TEST_PATH)

# Lấy khách chẵn làm Validation
lf_val = lf_test.filter((pl.col('mapped_user_id') % 2) == 0)
valid_users = lf_val.select('mapped_user_id').unique()

lf_cands_val = lf_cands.join(valid_users, on='mapped_user_id', how='inner')

lf_labels = (
    lf_cands_val.select(['mapped_user_id', 'mapped_item_id'])
    .join(
        lf_val.select(['mapped_user_id', 'mapped_item_id']).unique().with_columns(pl.lit(1).alias('label').cast(pl.Int8)),
        on=['mapped_user_id', 'mapped_item_id'],
        how='left'
    )
    .with_columns(pl.col('label').fill_null(0).cast(pl.Int8))
)

df_all_labels = lf_labels.collect(engine="streaming")

df_positives = df_all_labels.filter(pl.col('label') == 1)
df_negatives = df_all_labels.filter(pl.col('label') == 0)
df_sampled_negatives = (
    df_negatives
    .sample(fraction=1.0, shuffle=True, seed=42) 
    .group_by('mapped_user_id', maintain_order=True) 
    .head(NEGATIVE_RATIO)
)
df_labels = pl.concat([df_positives, df_sampled_negatives])
print(f"Tổng số mẫu dương tính (Nhãn 1 - Hit): {df_positives.height:,}")
print(f"Tổng số mẫu huấn luyện Ranker (Dòng): {df_labels.height:,}")

print("Bước 5: Nối Đặc trưng theo từng khối (Tối ưu RAM tuyệt đối)...")

print("-> Đang nạp các bảng thống kê và Meta vào bộ nhớ...")
df_meta_mem = lf_meta.collect()
df_user_stats_mem = user_stats.collect()
df_item_stats_mem = item_stats.collect()

lf_ranks = lf_cands.select(['mapped_user_id', 'mapped_item_id', 'sasrec_rank', 'lightgcn_rank'])

total_rows = df_labels.height
TEMP_DIR = "feat_chunks_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

for start_idx in range(0, total_rows, CHUNK_SIZE):
    end_idx = min(start_idx + CHUNK_SIZE, total_rows)
    print(f"-> Đang xử lý khối {start_idx:,} đến {end_idx:,}...")
    
    chunk_df = df_labels.slice(start_idx, CHUNK_SIZE)
    
    unique_users = chunk_df['mapped_user_id'].unique().to_list()
    df_ranks_chunk = lf_ranks.filter(pl.col('mapped_user_id').is_in(unique_users)).unique(subset=['mapped_user_id', 'mapped_item_id']).collect()
    
    chunk_processed = (
        chunk_df.lazy()
        .join(df_ranks_chunk.lazy(), on=['mapped_user_id', 'mapped_item_id'], how='left')
        .join(df_meta_mem.lazy(), on='mapped_item_id', how='left')
        .join(df_user_stats_mem.lazy(), on='mapped_user_id', how='left')
        .join(df_item_stats_mem.lazy(), on='mapped_item_id', how='left')
        .with_columns([
            ((201.0 - pl.col('sasrec_rank').cast(pl.Float32)).clip(lower_bound=0.0)).fill_null(0.0).alias('sasrec_score'),
            ((201.0 - pl.col('lightgcn_rank').cast(pl.Float32)).clip(lower_bound=0.0)).fill_null(0.0).alias('lightgcn_score'),
            
            # pl.col('price').fill_null(0.0),
            # pl.col('average_rating').fill_null(0.0),
            pl.col('rating_number').fill_null(0),
            pl.col('user_total_actions').fill_null(0),
            pl.col('item_total_sales').fill_null(0),
            pl.col('user_avg_rating_given').fill_null(3.0)
        ])
        .drop(['sasrec_rank', 'lightgcn_rank'])
        .collect()
    )
    
    chunk_processed.write_parquet(f"{TEMP_DIR}/chunk_{start_idx}.parquet")
    
    del chunk_df, df_ranks_chunk, chunk_processed, unique_users
    gc.collect()

print("-> Đang hợp nhất các khối thành file Features hoàn chỉnh...")
pl.scan_parquet(f"{TEMP_DIR}/*.parquet").with_columns([
    pl.col('store').cast(pl.Utf8),
    pl.col('main_category').cast(pl.Utf8)
]).sink_parquet(FEAT_OUT)

shutil.rmtree(TEMP_DIR)
del df_labels, lf_train, lf_meta, lf_cands, user_stats, item_stats
del df_meta_mem, df_user_stats_mem, df_item_stats_mem
gc.collect()

print(f"Hoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: {FEAT_OUT}")

Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...
Bước 2: Tính toán đặc trưng thống kê & Trọng số...
Bước 3: Tối ưu bộ nhớ Meta Data...
Bước 4: Đánh nhãn trực tiếp trên tập Ứng viên (Chống Data Leakage)...
Tổng số mẫu dương tính (Nhãn 1 - Hit): 49,043
Tổng số mẫu huấn luyện Ranker (Dòng): 2,113,099
Bước 5: Nối Đặc trưng theo từng khối (Tối ưu RAM tuyệt đối)...
-> Đang nạp các bảng thống kê và Meta vào bộ nhớ...
-> Đang xử lý khối 0 đến 2,113,099...
-> Đang hợp nhất các khối thành file Features hoàn chỉnh...
Hoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: /kaggle/working/features.parquet


In [4]:
print(lf_ranks.head(200).collect())

shape: (200, 4)
┌────────────────┬────────────────┬─────────────┬───────────────┐
│ mapped_user_id ┆ mapped_item_id ┆ sasrec_rank ┆ lightgcn_rank │
│ ---            ┆ ---            ┆ ---         ┆ ---           │
│ i64            ┆ i32            ┆ i16         ┆ i16           │
╞════════════════╪════════════════╪═════════════╪═══════════════╡
│ 0              ┆ 567361         ┆ null        ┆ 1             │
│ 0              ┆ 251896         ┆ null        ┆ 2             │
│ 0              ┆ 467407         ┆ null        ┆ 3             │
│ 0              ┆ 218887         ┆ null        ┆ 4             │
│ 0              ┆ 219449         ┆ null        ┆ 5             │
│ …              ┆ …              ┆ …           ┆ …             │
│ 1              ┆ 169038         ┆ null        ┆ 49            │
│ 1              ┆ 152317         ┆ 50          ┆ null          │
│ 1              ┆ 615751         ┆ null        ┆ 50            │
│ 1              ┆ 468160         ┆ 51          ┆ null      